# 02 · 모델 학습 (Training)

전처리된 **정상(train) 데이터**로 MTadGAN 학습 → `./checkpoints/` 저장.
설정은 **`config.py`**에서 불러온다(하이퍼파라미터는 거기서 관리).

**PCA 모드**(`config.py`의 `pca_mode`):
- `'consistent'` (정합) : 학습에서 fit한 PCA·스케일러를 **저장** → 테스트가 재사용
- `'original'`  (원본) : 저장하지 않음(테스트가 각자 fit_transform)

In [1]:
from function_def import *
from parameter import CFG
import os, numpy as np, pandas as pd, joblib
import tensorflow as tf

## 하이퍼파라미터 로드 (+ 필요시 이 셀에서 임시 오버라이드)

`config.py`를 고치면 학습·테스트에 동시 반영된다.
이 노트북에서만 잠깐 바꾸고 싶으면 아래 주석처럼 CFG 값을 덮어쓴다.

In [2]:
# 예: 이 실행에서만 win_size/epochs 바꾸기 (파생값도 다시 계산하려면 make_config 사용)
# from config import make_config
# CFG = make_config(win_size=30, epochs=50)

win_size      = CFG['win_size']
features_dim  = CFG['features_dim']
feat_dim      = CFG['feat_dim']
latent_dim    = CFG['latent_dim']
batch_size    = CFG['batch_size']
n_critic      = CFG['n_critic']
epochs        = CFG['epochs']
learning_rate = CFG['learning_rate']
k_size        = CFG['k_size']
lstm_units    = CFG['lstm_units']
drop_gen      = CFG['dropout_rate_gen']
crit_filters  = CFG['critic_filters']
crit_drop     = CFG['critic_dropout']
diffs_n, lags_n, smooth_n = CFG['diffs_n'], CFG['lags_n'], CFG['smooth_n']
pca_mode = CFG['pca_mode']

shape                   = CFG['shape']
encoder_input_shape     = CFG['encoder_input_shape']
encoder_reshape_shape   = CFG['encoder_reshape_shape']
generator_input_shape   = CFG['generator_input_shape']
generator_reshape_shape = CFG['generator_reshape_shape']
critic_x_input_shape    = CFG['critic_x_input_shape']
critic_z_input_shape    = CFG['critic_z_input_shape']

ckpt_dir = CFG['ckpt_dir']; os.makedirs(ckpt_dir, exist_ok=True)
print("win_size=%d features_dim=%d k_size=%d epochs=%d pca_mode=%s"
      % (win_size, features_dim, k_size, epochs, pca_mode))

win_size=10 features_dim=3 k_size=2 epochs=30 pca_mode=original


## 설정 (학습 데이터 경로)

In [3]:
TRAIN_CSV = './data/preprocessed/train/1000_chg.csv'
print("TRAIN_CSV:", TRAIN_CSV)

TRAIN_CSV: ./data/preprocessed/train/1000_chg.csv


## 1) 로드 → featurize → PCA(fit)

208차원(상관 높음) → **PCA로 축소**(공분산 행렬 고유벡터로 투영).
정합 모드면 여기서 fit한 `pca`·`scaler`를 **저장**해 테스트가 재사용하게 한다.

In [4]:
df_train_0 = pd.read_csv(TRAIN_CSV)
data_1 = diff_smooth_df(df_train_0, lags_n, diffs_n, smooth_n)

pca = PCA(n_components=features_dim)
data = pca.fit_transform(data_1)          # 학습 데이터로 축 확정(fit) + 변환

df_1 = []
for i in range(len(data)):
    row = [i + 1] + [data[i][jj] for jj in range(features_dim)]
    df_1.append(row)
df = pd.DataFrame(df_1)
df.columns = ['date'] + ['pca_%s' % str(i) for i in range(1, features_dim + 1)]
print("After PCA:", df.shape)



After PCA: (6009, 4)


In [ ]:
X, index = time_segments_aggregate(df, interval=1, time_column='date')
X = SimpleImputer().fit_transform(X)

scaler = MinMaxScaler(feature_range=(-1, 1))
X = scaler.fit_transform(X)               # 학습 데이터로 스케일 범위 확정(fit)

# 정합 모드: 학습에서 fit한 pca/scaler 저장 -> 테스트가 transform 으로 재사용
if pca_mode == 'consistent':
    joblib.dump(pca, CFG['pca_path'])
    joblib.dump(scaler, CFG['scaler_path'])
    print("saved pca/scaler ->", CFG['pca_path'], CFG['scaler_path'])
else:
    print("original 모드: pca/scaler 저장 안 함(테스트가 각자 fit_transform)")

X, y, X_index, y_index = rolling_window_sequences(
    X, index, window_size=win_size, target_size=1, step_size=1, target_column=0)
print("after window:", X.shape)

original 모드: pca/scaler 저장 안 함(테스트가 각자 fit_transform)
after window: (5999, 10, 3)


## 2) 네트워크 생성 (config 하이퍼파라미터를 build 함수에 전달)

In [6]:
encoder   = build_encoder_layer(encoder_input_shape, encoder_reshape_shape,
                                win_size=win_size, latent_dim=latent_dim)
generator = build_generator_layer(generator_input_shape, generator_reshape_shape,
                                  win_size=win_size, features_dim=features_dim,
                                  lstm_units=lstm_units, dropout_rate=drop_gen)
critic_x  = build_critic_x_layer(critic_x_input_shape, k_size=k_size,
                                 filters=crit_filters, dropout_rate=crit_drop)
critic_z  = build_critic_z_layer(critic_z_input_shape)
optimizer = tf.keras.optimizers.Adam(learning_rate)
print("networks & optimizer ready")

networks & optimizer ready


## 3) 합성 모델 구성

In [7]:
z = Input(shape=(latent_dim, 1)); x = Input(shape=shape)
x_ = generator(z); z_ = encoder(x)
critic_x_model = Model([x, z],
    [critic_x(x), critic_x(x_), RandomWeightedAverage(batch_size)([x, x_])])
critic_z_model = Model([x, z],
    [critic_z(z), critic_z(z_), RandomWeightedAverage(batch_size)([z, z_])])
z_gen = Input(shape=(latent_dim, 1)); x_gen = Input(shape=shape)
x_gen_ = generator(z_gen); z_gen_ = encoder(x_gen); x_gen_rec = generator(z_gen_)
encoder_generator_model = Model([x_gen, z_gen],
    [critic_x(x_gen_), critic_z(z_gen_), x_gen_rec])
print("composite models ready")

composite models ready


## 4) (선택) 기존 가중치 이어서 학습

In [ ]:
#for name, model in [('critic_x_model', critic_x_model),
 #                   ('critic_z_model', critic_z_model),
  #                  ('encoder_generator_model', encoder_generator_model)]:
   # p = os.path.join(ckpt_dir, name + '.h5')
    #if os.path.isfile(p):
     #   model.load_weights(p); print("loaded:", name)
    #else:
     #    print("fresh:", name)

loaded: critic_x_model
loaded: critic_z_model
loaded: encoder_generator_model


## 5) 학습 루프

판별자를 생성자보다 `n_critic`배 자주 업데이트(WGAN). GAN 특성상 실행마다 결과가 조금씩 달라진다.
재현이 필요하면 아래 seed 고정 주석을 활성화.

In [9]:
# 재현용 seed 고정 (선택)
# np.random.seed(42); tf.random.set_seed(42)

X = X.reshape((-1, shape[0], feat_dim))
X_ = np.copy(X)
fake  =  np.ones((batch_size, 1), dtype=np.float32)
valid = -np.ones((batch_size, 1), dtype=np.float32)
delta =  np.ones((batch_size, 1), dtype=np.float32)

for epoch in range(1, epochs + 1):
    np.random.shuffle(X_)
    g_loss, cx_loss, cz_loss = [], [], []
    mb_size = batch_size * n_critic
    num_mb = int(X_.shape[0] // mb_size)
    for i in range(num_mb):
        mb = X_[i * mb_size:(i + 1) * mb_size]
        critic_x.trainable = True;  critic_z.trainable = True
        generator.trainable = False; encoder.trainable = False
        for j in range(n_critic):
            xb = mb[j * batch_size:(j + 1) * batch_size]
            zb = np.random.normal(size=(batch_size, latent_dim, 1))
            cx_loss.append(critic_x_train_on_batch(xb, zb, valid, fake, delta,
                                                   critic_x_model, critic_x, optimizer))
            cz_loss.append(critic_z_train_on_batch(xb, zb, valid, fake, delta,
                                                   critic_z_model, critic_z, optimizer))
        critic_x.trainable = False; critic_z.trainable = False
        generator.trainable = True;  encoder.trainable = True
        g_loss.append(enc_gen_train_on_batch(xb, zb, valid,
                                             encoder_generator_model, optimizer))
    print('Epoch {}/{}, [Dx {}] [Dz {}] [G {}]'.format(
        epoch, epochs, np.mean(np.array(cx_loss), axis=0),
        np.mean(np.array(cz_loss), axis=0), np.mean(np.array(g_loss), axis=0)))

Epoch 1/30, [Dx 0.9793111085891724] [Dz -3.721421957015991] [G 1.8426004648208618]
Epoch 2/30, [Dx 1.3499586582183838] [Dz -3.27384090423584] [G 1.3898512125015259]
Epoch 3/30, [Dx 0.4016307294368744] [Dz -1.4492050409317017] [G -2.6473381519317627]
Epoch 4/30, [Dx 1.0395665168762207] [Dz -3.0107436180114746] [G -8.028398513793945]
Epoch 5/30, [Dx 1.1047673225402832] [Dz -3.856213331222534] [G 4.413049697875977]
Epoch 6/30, [Dx 0.7857500910758972] [Dz -2.281996011734009] [G 5.049746990203857]
Epoch 7/30, [Dx 0.9927059412002563] [Dz -2.614682674407959] [G 2.605365037918091]
Epoch 8/30, [Dx 1.433598518371582] [Dz -1.4238067865371704] [G -5.085400104522705]
Epoch 9/30, [Dx 1.6632542610168457] [Dz -1.7948105335235596] [G -9.568792343139648]
Epoch 10/30, [Dx 1.194463849067688] [Dz -2.3393661975860596] [G -8.523365020751953]
Epoch 11/30, [Dx 0.880422830581665] [Dz -1.382910966873169] [G 0.37011826038360596]
Epoch 12/30, [Dx 0.7854929566383362] [Dz -1.2709325551986694] [G -0.6283261775970459]

## 6) 체크포인트 저장

In [10]:
critic_x_model.save_weights(os.path.join(ckpt_dir, 'critic_x_model.h5'), save_format='h5')
critic_z_model.save_weights(os.path.join(ckpt_dir, 'critic_z_model.h5'), save_format='h5')
encoder_generator_model.save_weights(os.path.join(ckpt_dir, 'encoder_generator_model.h5'), save_format='h5')
print("checkpoints saved to", ckpt_dir)

checkpoints saved to checkpoints
